# Week 5 — MCMC, Degeneracy and Hierarchical Population Inference

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../07_MCMC/MCMC.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Explain why posterior sampling is required when analytic integration is unavailable.
- Implement and diagnose a basic MCMC sampler.
- Use trace plots, autocorrelation and effective sample size rather than relying on corner plots alone.
- Formulate a hierarchical model that separates object-level parameters from population hyperparameters.
- Explain partial pooling and shrinkage in astronomical population inference.

## 1. Why sampling?


A posterior in a realistic astronomical model may have tens, hundreds or thousands of dimensions. We often need expectations such as

$$
E[f(\theta)\mid D]=\int f(\theta)p(\theta\mid D)\,d\theta
$$

rather than only the maximum. MCMC constructs a correlated sequence whose stationary distribution is the posterior, allowing integrals to be approximated by sample averages.

The samples are not independent. The statistical quality of an MCMC run therefore depends on mixing and autocorrelation, not simply the raw number of stored draws.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Minimal random-walk Metropolis

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(2026)

def logtarget(x):
    return -0.5*(x/1.5)**2

x=0.; chain=[]; accepted=0
for i in range(25000):
    xp=x+rng.normal(0,.8)
    if np.log(rng.random()) < logtarget(xp)-logtarget(x):
        x=xp; accepted+=1
    chain.append(x)
chain=np.asarray(chain)
print('acceptance:',accepted/len(chain))
plt.plot(chain[:3000]); plt.xlabel('iteration'); plt.ylabel('x'); plt.show()

## 2. Metropolis-Hastings and detailed balance


Given a current state $\theta$, propose $\theta'\sim q(\theta'\mid\theta)$ and accept it with probability

$$
a=\min\left[1,
\frac{p(\theta'\mid D)q(\theta\mid\theta')}
{p(\theta\mid D)q(\theta'\mid\theta)}
\right].
$$

For a symmetric random-walk proposal the proposal ratio cancels. A proposal scale that is far too small produces slow random-walk exploration; one that is far too large produces many rejections.

MCMC is therefore a computational method whose tuning and geometry matter. More sophisticated samplers improve this geometry, but the diagnostic principles remain.

### Worked computational demonstration — Simple autocorrelation estimate

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def acf(x,maxlag=100):
    x=np.asarray(x)-np.mean(x)
    var=np.dot(x,x)
    return np.array([np.dot(x[:-k],x[k:])/var if k else 1. for k in range(maxlag+1)])
a=acf(chain[2000:],100)
plt.plot(a); plt.axhline(0); plt.xlabel('lag'); plt.ylabel('ACF'); plt.show()
print('rough ESS:',len(chain[2000:])/(1+2*np.sum(a[1:][a[1:]>0])))

## 3. Diagnostics: convergence is a claim


A trace plot can reveal trends, sticking and multimodal transitions. The autocorrelation function shows how quickly a chain forgets its state. Effective sample size is roughly

$$
N_{\rm eff}\approx \frac{N}{1+2\sum_{k\ge1}\rho_k}.
$$

Multiple-chain diagnostics such as $\hat R$ compare within-chain and between-chain variation. No single diagnostic proves convergence, but several complementary diagnostics can expose failure.

A visually attractive corner plot is only a representation of the samples you obtained. If the chain did not explore the posterior correctly, the corner plot faithfully visualises the wrong sample.

### Worked computational demonstration — Hierarchical shrinkage toy

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
rng=np.random.default_rng(7)
true_mu,true_tau=0.0,0.5
theta=rng.normal(true_mu,true_tau,20)
sigma_obs=np.linspace(.1,1.0,20)
y=rng.normal(theta,sigma_obs)

# If population hyperparameters were known, posterior mean for Normal-Normal model:
w=1/sigma_obs**2
wp=1/true_tau**2
post_mean=(w*y+wp*true_mu)/(w+wp)
print('mean absolute raw error:',np.mean(np.abs(y-theta)))
print('mean absolute shrunk error:',np.mean(np.abs(post_mean-theta)))

## 4. Hierarchical models and partial pooling


Suppose every binary has an object-level parameter $\theta_i$ drawn from a population distribution controlled by hyperparameters $\Lambda$:

$$
\theta_i\sim p(\theta_i\mid\Lambda),\qquad
D_i\sim p(D_i\mid\theta_i).
$$

Then

$$
p(\Lambda,\{\theta_i\}\mid\{D_i\})
\propto p(\Lambda)\prod_i
p(\theta_i\mid\Lambda)p(D_i\mid\theta_i).
$$

This structure connects noisy individual measurements to population inference. Objects with weak data borrow statistical strength from the population; strongly measured objects remain data-driven. The resulting shrinkage is not an arbitrary correction but a consequence of the joint probabilistic model.

## 5. Population inference with simulations


The binary-SMBH catalogue gives us a rare luxury: true object-level parameters. We can use them to test hierarchical procedures under controlled conditions. But again, recovering the distribution used by the simulator is a methodological validation, not yet a measurement of the cosmic binary population.

A useful lab formulation is to add artificial measurement noise to a true latent variable, compare independent per-object estimates with a hierarchical model, and quantify bias and shrinkage.

## Deep dive — hierarchical inference through marginalisation

The hierarchical model can be written either with explicit object-level latent variables or after marginalising them. For object $i$,

$$
p(D_i\mid\Lambda)
=
\int p(D_i\mid\theta_i)p(\theta_i\mid\Lambda)\,d\theta_i.
$$

Then the population posterior is

$$
p(\Lambda\mid\{D_i\})
\propto
p(\Lambda)\prod_i p(D_i\mid\Lambda).
$$

This form clarifies that hierarchical inference does not require us to report every $\theta_i$, although latent-variable sampling can be computationally useful.

### Centered and non-centred parameterisations

In hierarchical Gaussian models, the representation

$$
\theta_i\sim N(\mu,\tau)
$$

is called centered. A non-centred representation writes

$$
z_i\sim N(0,1),\qquad
\theta_i=\mu+\tau z_i.
$$

The two are mathematically equivalent but can have very different posterior geometry and MCMC efficiency, especially when individual measurements are weak. This is a powerful lesson: computational difficulty can depend strongly on parameterisation even when the probability model is unchanged.

### Shrinkage is conditional, not arbitrary

Objects with large measurement error are pulled more strongly toward the inferred population distribution. Well-measured objects are affected less. This is exactly what one wants if the hierarchical model is correct, but it can be dangerous if the population model is too restrictive or contains unmodelled sub-populations.

A diagnostic is to examine shrinkage against measurement uncertainty and to test whether posterior predictive simulations reproduce the diversity of the population.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `07_MCMC/MCMC.ipynb`.
- Gelman et al., *Bayesian Data Analysis*.
- Foreman-Mackey et al. (emcee) for ensemble MCMC.

# Part II — Extended lecture development

## 13. Detailed balance and stationarity

A Markov transition kernel $K(\theta'\mid\theta)$ has target distribution
$\pi(\theta)$ as stationary when

$$
\pi(\theta')
=
\int
\pi(\theta)K(\theta'\mid\theta)d\theta.
$$

Metropolis–Hastings constructs a kernel satisfying detailed balance,

$$
\pi(\theta)K(\theta'\mid\theta)
=
\pi(\theta')K(\theta\mid\theta'),
$$

which is sufficient for stationarity.

This formalism explains why rejected proposals are not "wasted": remaining at
the current state is part of the transition kernel required to obtain the target
distribution.

## 14. Monte Carlo error is distinct from posterior width

Posterior uncertainty describes uncertainty about parameters under the model.
Monte Carlo error describes finite numerical uncertainty in estimating posterior
summaries from correlated samples.

If a posterior standard deviation is 0.5, that does not mean the Monte Carlo
error in its estimated mean is 0.5. Roughly,

$$
{\rm SE}_{MC}(\bar\theta)
\approx
\frac{\sigma_\theta}{\sqrt{N_{\rm eff}}}.
$$

A scientifically broad posterior can be estimated numerically very precisely,
and a narrow posterior can be estimated poorly if mixing is bad.

## 15. Gelman–Rubin-style multi-chain thinking

For several independent chains, compare within-chain variance with between-chain
variance. If chains have reached and explored the same stationary target, these
should agree.

The modern $\hat R$ diagnostic is close to 1 when chains are well mixed. Values
substantially above 1 indicate lack of convergence.

No threshold is magical. Trace plots, effective sample size, chain rank plots
and domain-specific checks should be considered together.

## 16. Multimodality: a physically important failure mode

Binary systems can naturally produce degeneracies: phase aliases, orientation
symmetries or multiple parameter combinations that yield similar light curves.

If a posterior has separated modes, reporting one mean and covariance can be
actively misleading.

Strategies include:
- better-informed proposals;
- parallel tempering or other advanced samplers;
- nested sampling;
- reparameterisation;
- explicit symmetry handling.

The first task is not to force a single mode but to understand whether
multimodality is physical, numerical or caused by model non-identifiability.

In [ ]:
# Toy bimodal target to illustrate mode trapping
rng=np.random.default_rng(2026)
def log_bimodal(x):
    # log of equal mixture N(-4,1) + N(4,1)
    a=stats.norm.logpdf(x,-4,1)
    b=stats.norm.logpdf(x,4,1)
    m=max(a,b)
    return m+np.log(.5*np.exp(a-m)+.5*np.exp(b-m))

def run(start,step=1,n=12000):
    x=float(start); out=[]
    for _ in range(n):
        xp=x+rng.normal(0,step)
        if np.log(rng.random())<log_bimodal(xp)-log_bimodal(x):
            x=xp
        out.append(x)
    return np.asarray(out)

a=run(-4); b=run(4)
print("chain means:",a.mean(),b.mean())
plt.plot(a[:3000],label="start -4")
plt.plot(b[:3000],label="start +4")
plt.legend(); plt.show()

## 17. Ensemble samplers and affine invariance

The `emcee` approach used in the Crete notebook employs multiple walkers and
moves designed to perform well under linear stretching/rotation of parameter
space. This reduces sensitivity to highly elongated Gaussian-like posteriors.

But affine invariance does not solve:
- disconnected modes;
- hard pathological boundaries;
- non-identifiability;
- very expensive likelihoods;
- model misspecification.

The correct attitude is: use an appropriate sampler, then diagnose it.

## 18. Hierarchical shrinkage derived explicitly

For

$$
\theta_i\sim N(\mu,\tau^2),\qquad
y_i\sim N(\theta_i,\sigma_i^2),
$$

the conditional posterior for $\theta_i$ with known $\mu,\tau$ is Normal.
Its mean is

$$
E[\theta_i\mid y_i]
=
\frac{y_i/\sigma_i^2+\mu/\tau^2}
{1/\sigma_i^2+1/\tau^2}.
$$

This is a precision-weighted average. When $\sigma_i\ll\tau$, the data
dominate. When $\sigma_i\gg\tau$, the population mean dominates.

This equation is the mathematical heart of partial pooling.

## 19. Population misspecification in hierarchical models

Hierarchical models borrow strength by assuming exchangeability within a
population model. If the true population contains distinct subgroups but the
model assumes one narrow Gaussian, the hierarchy can over-shrink objects toward
a fictitious centre.

Posterior predictive checks at the population level should examine:
- tails;
- multimodality;
- subgroup residuals;
- dependence on known context variables.

Partial pooling is only as trustworthy as the population model supplying the
pool.

## 20. Hierarchical model for noisy SMBH period estimates

Imagine each binary has latent log-period $\theta_i=\log P_i$ and a noisy
measurement $\hat\theta_i$ with known error $s_i$:

$$
\theta_i\sim N(\mu_P,\tau_P^2),
\qquad
\hat\theta_i\sim N(\theta_i,s_i^2).
$$

The scientific target may be $(\mu_P,\tau_P)$, not the individual periods.

With simulated data we can:
1. hide true periods;
2. generate noisy measurements;
3. fit the hierarchical model;
4. recover both population and object-level quantities;
5. check coverage using truth.

This is a direct bridge from statistical teaching to survey population work.

## 21. Selection will alter the hierarchy

If detection depends on $\theta_i$, the population model for detected objects
is no longer simply $p(\theta_i\mid\Lambda)$.

The Week-6 likelihood will contain

$$
\alpha(\Lambda)
=
\int p_{\rm det}(\theta)p(\theta\mid\Lambda)d\theta.
$$

Hierarchical inference without selection is therefore only half the population
problem.

# Part III — Worked hierarchical reasoning

## Problem A — why independent fits can bias a population

Suppose each source has noisy estimate

$$
\hat\theta_i\sim N(\theta_i,s_i^2).
$$

If we histogram $\hat\theta_i$, measurement noise broadens the observed
distribution. Estimating population scatter directly from that histogram
confounds intrinsic dispersion with measurement error.

A hierarchical model instead integrates over each latent $\theta_i$ and can
separate the two sources of spread.

This is one reason population inference should not simply histogram catalogue
point estimates.

## Problem B — funnel geometry

Consider

$$
\tau\sim p(\tau),\qquad
\theta_i\sim N(0,\tau^2).
$$

When $\tau$ is small, all $\theta_i$ must lie in a very narrow region; when
$\tau$ is large, they can spread widely. The joint posterior can resemble a
funnel.

A non-centred representation

$$
z_i\sim N(0,1),\qquad\theta_i=\tau z_i
$$

often makes the geometry easier for a sampler.

Draw this geometry on the board before introducing the term
"non-centred parameterisation." Students remember the computational reason
better than the terminology.

## Problem C — posterior predictive population check

After fitting population hyperparameters $\Lambda$:

1. draw $\Lambda^{(s)}$ from the posterior;
2. draw new latent $\theta_{\rm rep}^{(s)}\sim
   p(\theta\mid\Lambda^{(s)})$;
3. generate mock observations if needed;
4. compare tails, multimodality and subgroup structure with the observed sample.

This checks the population model, not only individual-object fits.

# Part IV — Additional MCMC/hierarchy seminar

## Thinning is usually not a cure for autocorrelation

A common practice is to retain every $k$-th MCMC draw. This reduces stored
autocorrelation but throws away samples. For estimating posterior expectations,
keeping all post-warm-up samples is usually more efficient unless storage or a
specific downstream format requires thinning.

The solution to poor mixing is better sampling/parameterisation, not deleting
most of the chain.

## Monte Carlo standard error should be reported for delicate quantities

If a posterior tail probability is scientifically important, estimate whether
the chain contains enough effective draws in that tail. A posterior probability
of 0.001 estimated from only a few effective tail samples is numerically noisy
even if the central posterior looks stable.

### Hierarchical exchangeability

Writing

$$
\theta_i\sim p(\theta\mid\Lambda)
$$

assumes objects are exchangeable conditional on $\Lambda$. If WFD and DDF
objects are generated/observed under meaningfully different structures, the
hierarchy may need group-specific terms rather than pretending all objects
share one population relation.

The hierarchy should mirror scientific grouping, not merely provide shrinkage.